# Step 1: Shrinking an AI Model — PyTorch to ONNX & INT8 (Beginner-Friendly Guide)

### What We Are Building Today (In Plain English)
Imagine you have a smart camera that needs to recognize objects **offline** (no internet, limited memory, needs to be fast).
Today we will take a famous pretrained image-recognition AI (`ResNet-18`) and run it through the **3 core steps of Edge AI engineering**:
1. **Convert PyTorch -> ONNX:**
   - **PyTorch** is like a Microsoft Word `.docx` file—awesome for building and editing an AI model, but heavy to ship.
   - **ONNX (Open Neural Network Exchange)** is like a universal `.pdf` file—a standard blueprint of the AI's math that can run cleanly on almost any computer, phone, or smart camera.
   - **ONNX Runtime (ORT)** is the super-fast "PDF reader" engine that executes that `.onnx` blueprint.
2. **Shrink the numbers from FP32 -> INT8 (Quantization):**
   - Normally, AI stores every learned number as an **FP32** (32-bit decimal, like `0.48291734`) — very detailed, like a huge **RAW photo** (4 bytes per number).
   - **INT8 Quantization** rounds those decimals into simple 8-bit whole numbers (like `0` to `255` or `-128` to `127`) — like turning a RAW photo into a crisp **JPEG** (1 byte per number, **~4x smaller**!).
3. **Measure Everything Like a Pro (Benchmarking):**
   - Did it actually get **smaller** (Megabytes)?
   - Did it actually get **faster** (Milliseconds per photo)?
   - Did it stay **accurate** (does it still guess the same objects as before)?

### How to Run This in Google Colab
1. Open [Google Colab](https://colab.research.google.com/) -> `File -> Upload notebook` -> upload this `.ipynb` file.
2. Keep the runtime on **CPU** (`Runtime -> Change runtime type -> CPU`) so you clearly see real INT8 speedups on standard processors.
3. Run cells from top to bottom and read each **Simple Explanation** box before running the code beneath it.

## 0. Setup & Checking Our Computer Specs

### Simple Explanation
Imagine someone brags: *"My car finished the race in 10 seconds!"* Your first question would be: *"Wait—on a race track or uphill in mud?"*

In Edge AI, speed (**latency**) depends completely on **what processor (CPU/GPU)** and **how many CPU cores/threads** ran the test. So every good benchmark always prints the exact computer specs first.
- **CPU Cores / Threads:** Think of cores as chefs in a kitchen working side by side.
- **RAM:** The kitchen counter space available to hold the model while it works.
- **ORT Providers:** The engines ONNX Runtime can use (e.g., `CPUExecutionProvider` for CPU, `CUDAExecutionProvider` for NVIDIA GPUs).

In [8]:
import os
import time
import json
import inspect
import platform
from pathlib import Path
from collections import Counter

import numpy as np
import psutil
import torch
import torchvision
from torchvision import models, transforms
import onnx # type:ignore
import onnxruntime as ort # type:ignore

# Fix random seeds so results are repeatable
torch.manual_seed(42)
np.random.seed(42)

# Folder inside Colab where we will save our exported and shrunk model files
ARTIFACT_DIR = Path("step01_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python        : {platform.python_version()}")
print(f"PyTorch       : {torch.__version__}")
print(f"ONNX          : {onnx.__version__}")
print(f"ONNX Runtime  : {ort.__version__}")
print(f"CPU Cores     : {os.cpu_count()} logical | RAM: {psutil.virtual_memory().total / (1024**3):.1f} GB")
print(f"ORT Providers : {ort.get_available_providers()}")
print(f"CUDA Available: {torch.cuda.is_available()}")

Python        : 3.13.16
PyTorch       : 2.14.1+cu130
ONNX          : 1.23.2
ONNX Runtime  : 1.32.0
CPU Cores     : 2 logical | RAM: 12.7 GB
ORT Providers : ['AzureExecutionProvider', 'CPUExecutionProvider']
CUDA Available: True


## 1. Baseline: Load Pretrained `ResNet-18` (Our Starting Model)

### Simple Explanation
- **What is `ResNet-18`?** It's a classic, lightweight "AI eye" (a **CNN** — *Convolutional Neural Network*) already trained on **ImageNet** (over 1 million photos across 1,000 everyday categories like dogs, cars, coffee mugs, and keyboards).
- **What are Parameters / Weights?** Think of them as **11.7 million tiny volume knobs** inside the AI's brain. Each knob holds a learned decimal number.
- **Why ~45 MB?** Each knob is stored in **FP32** (4 bytes). So $11.69\text{ million} \times 4\text{ bytes} \approx 44.6\text{ MB}$ on disk.
- **Why call `pt_model.eval()`?** AI models have two modes: **Study Mode (`train`)** and **Exam Mode (`eval`)**. In Study Mode, layers like `BatchNorm` and `Dropout` keep changing slightly to help learning. Before exporting or testing, you **always** switch to Exam Mode (`eval()`) so every answer is frozen and consistent.

In [9]:
# Load pretrained ResNet-18 with official ImageNet-1K weights
weights = models.ResNet18_Weights.DEFAULT
pt_model = models.resnet18(weights=weights)
pt_model.eval()  # Switch to Exam Mode (freeze BatchNorm & Dropout)

# List of 1,000 human-readable object names (e.g. 'golden retriever', 'espresso')
IMAGENET_CLASSES = weights.meta["categories"]

# Count total learned numbers (parameters) and check baseline file size
num_params = sum(p.numel() for p in pt_model.parameters())
pt_path = ARTIFACT_DIR / "resnet18_fp32.pth"
torch.save(pt_model.state_dict(), pt_path)
pt_size_mb = pt_path.stat().st_size / (1024 * 1024)

print(f"Parameters (learned knobs) : {num_params:,}")
print(f"PyTorch FP32 File Size     : {pt_size_mb:.2f} MB")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 167MB/s]


Parameters (learned knobs) : 11,689,512
PyTorch FP32 File Size     : 44.67 MB


## 2. Prepare Real Photos (For Calibration & Testing)

### Simple Explanation
- **What is a Tensor & Shape `(1, 3, 224, 224)`?** Computers don't "see" photos like eyes do—they see a 4D grid (called a **Tensor**) of pixel numbers:
  - `1` = **Batch size** (1 photo at a time)
  - `3` = **Color channels** (Red, Green, Blue)
  - `224, 224` = **Height and Width** in pixels
- **Why use real photos instead of random TV-static noise?** Later, when we shrink the model to 8-bit whole numbers, we need to show it realistic pictures (**Calibration Set** = 100 photos) so it learns what normal brightness and feature numbers look like. Then we test its accuracy on 100 **brand-new** photos (**Evaluation Set**) it never saw during calibration.

In [10]:
# Standard ImageNet image prep: resize to 224x224, turn pixels into 0..1 floats, normalize colors
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# Download CIFAR-10 real photos (fast ~3 seconds on Colab)
cifar_test = torchvision.datasets.CIFAR10(
    root="./colab_data",
    train=False,
    download=True,
    transform=preprocess,
)

CALIB_COUNT = 100
EVAL_COUNT = 100

# Convert photos to NumPy arrays of shape (1, 3, 224, 224) in FP32
calib_images = [cifar_test[i][0].unsqueeze(0).numpy().astype(np.float32) for i in range(CALIB_COUNT)]
eval_images = [cifar_test[i][0].unsqueeze(0).numpy().astype(np.float32) for i in range(CALIB_COUNT, CALIB_COUNT + EVAL_COUNT)]

print(f"Calibration photos : {len(calib_images)} | shape: {calib_images[0].shape} | dtype: {calib_images[0].dtype}")
print(f"Evaluation photos  : {len(eval_images)}")

100%|██████████| 170M/170M [22:17<00:00, 128kB/s]  


Calibration photos : 100 | shape: (1, 3, 224, 224) | dtype: float32
Evaluation photos  : 100


## 3. Convert PyTorch -> Universal ONNX (`resnet18_fp32.onnx`) & Peek Inside

### Simple Explanation of Every Parameter Below
- **Computation Graph (DAG) & Nodes:** Think of an `.onnx` file as a **cooking recipe flowchart**. Each box in the flowchart is a **Node** (one math step like `Conv`, `Relu`, `Add`, or `Gemm`).
- **`dummy_input`:** To record the recipe, PyTorch sends one fake example image through the model and writes down every math step it touches.
- **`opset_version=17`:** The "dictionary edition" of ONNX math commands. Version 17 is widely supported by almost every engine (ONNX Runtime, NVIDIA TensorRT, Qualcomm AI Hub).
- **`do_constant_folding=True` (Operator Fusion):** A free speed trick! If a recipe says *"multiply by 2, then multiply by 3"*, PyTorch automatically merges it into *"multiply by 6"*. In `ResNet-18`, PyTorch folds **`Conv2d + BatchNorm2d`** into a single `Conv` node—that's why you will see **zero** `BatchNormalization` nodes below!
- **`dynamic_axes`:** Tells ONNX: *"Don't hard-lock the recipe to only 1 photo at a time—let me pass any batch size (1, 4, 16...) later."*

In [ ]:
onnx_fp32_path = ARTIFACT_DIR / "resnet18_fp32.onnx"
dummy_input = torch.randn(1, 3, 224, 224, dtype=torch.float32)

export_kwargs = dict(
    export_params=True,          # Pack the learned weights inside the .onnx file
    opset_version=17,            # Standard ONNX math dictionary version
    do_constant_folding=True,    # Merge Conv + BatchNorm for free speedup
    input_names=["input"],       # Clean name for input tensor
    output_names=["logits"],     # Clean name for raw output scores
    dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
)
# Use classic TorchScript exporter on PyTorch 2.5+ for smoothest ONNX Runtime quantization compatibility
if "dynamo" in inspect.signature(torch.onnx.export).parameters:
    export_kwargs["dynamo"] = False

with torch.no_grad():  # Turn off gradient tracking (we aren't training, saves memory)
    torch.onnx.export(pt_model, dummy_input, str(onnx_fp32_path), **export_kwargs)

# Run ONNX's built-in validator to ensure the blueprint has no broken wires
onnx_model = onnx.load(str(onnx_fp32_path))
onnx.checker.check_model(onnx_model)

# Count how many times each math step (operator) appears in the blueprint
op_counts = Counter(node.op_type for node in onnx_model.graph.node)
onnx_fp32_size_mb = onnx_fp32_path.stat().st_size / (1024 * 1024)

print(f"ONNX export valid! File size: {onnx_fp32_size_mb:.2f} MB")
print(f"Total math steps (nodes)    : {len(onnx_model.graph.node)}")
print(f"Math step breakdown         : {dict(op_counts)}")
print("\nCool check: Notice ZERO BatchNormalization nodes — PyTorch fused them into Conv!")

## 4. Sanity Check: Does Our ONNX Copy Give the Exact Same Answer as PyTorch?

### Simple Explanation
Before you try to shrink a PDF, you first check that the PDF didn't scramble any words from your Word document!
- **Logits:** The 1,000 raw scores `ResNet-18` outputs (one score per object class; higher score = more confident).
- **Numerical Equivalence (`max_abs_diff < 1e-4`):** We feed the *exact same photo* to PyTorch and to ONNX Runtime, subtract their 1,000 scores, and confirm the biggest difference is tiny (less than `0.0001`, which is just normal floating-point rounding).
- **Fair Thread Count (`NUM_THREADS`):** We give both PyTorch and ONNX Runtime the exact same number of CPU threads so our upcoming speed race is 100% fair.

In [ ]:
def create_ort_session(model_path: Path, num_threads: int = 2) -> ort.InferenceSession:
    """Open an .onnx file inside ONNX Runtime with full graph optimizations and fixed thread count."""
    sess_options = ort.SessionOptions()
    sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess_options.intra_op_num_threads = num_threads
    sess_options.inter_op_num_threads = 1
    return ort.InferenceSession(
        str(model_path),
        sess_options=sess_options,
        providers=["CPUExecutionProvider"],
    )

# Lock both PyTorch and ONNX Runtime to the same number of CPU threads for a fair speed test
NUM_THREADS = min(2, os.cpu_count() or 1)
torch.set_num_threads(NUM_THREADS)

ort_fp32_sess = create_ort_session(onnx_fp32_path, num_threads=NUM_THREADS)

# Feed the same photo to both engines and compare their 1,000 raw scores
sample_np = eval_images[0]
with torch.no_grad():
    pt_out = pt_model(torch.from_numpy(sample_np)).numpy()
ort_out = ort_fp32_sess.run(None, {"input": sample_np})[0]

max_abs_diff = float(np.max(np.abs(pt_out - ort_out)))
print(f"Max Score Difference (PyTorch vs ONNX FP32): {max_abs_diff:.2e}")
assert max_abs_diff < 1e-4, f"Export mismatch too large: {max_abs_diff}"
print("PASS: ONNX FP32 produces the exact same answers as PyTorch FP32!")

## 5. Shrinking Experiment A: Dynamic INT8 (`quantize_dynamic`)

### Simple Explanation (Watch Out for This Famous Beginner Surprise!)
There are two main ways to shrink an already-trained model (called **PTQ** = *Post-Training Quantization*):
1. **Dynamic Quantization (`quantize_dynamic`):** Shrinks weights ahead of time, and guesses how to round the live numbers **on the fly** while running—so you don't need calibration photos.
2. **Static Quantization (`quantize_static`):** Measures real photos ahead of time (calibration) and locks in the rounding rules for **every** layer.

Also, `quant_pre_process` below is simply a "cleanup step" that fills in missing tensor shapes and tidies the `.onnx` blueprint before quantization.

**Make a prediction before running this cell:** Will `quantize_dynamic` shrink `ResNet-18` from ~44.6 MB down to ~11 MB? Let's run it and see!

In [ ]:
from onnxruntime.quantization import (
    quantize_dynamic,
    quantize_static,
    CalibrationDataReader,
    CalibrationMethod,
    QuantFormat,
    QuantType,
)
from onnxruntime.quantization.shape_inference import quant_pre_process

# Step 5.1: Tidy up the FP32 graph (fill in tensor shapes) before quantizing
onnx_prep_path = ARTIFACT_DIR / "resnet18_prep.onnx"
try:
    quant_pre_process(str(onnx_fp32_path), str(onnx_prep_path), auto_merge=True)
except Exception as e:
    print(f"Symbolic shape inference fallback ({e}); retrying with skip_symbolic_shape=True...")
    quant_pre_process(str(onnx_fp32_path), str(onnx_prep_path), skip_symbolic_shape=True)

# Step 5.2: Apply Dynamic INT8 quantization
onnx_dyn_int8_path = ARTIFACT_DIR / "resnet18_dynamic_int8.onnx"
quantize_dynamic(
    model_input=str(onnx_prep_path),
    model_output=str(onnx_dyn_int8_path),
    weight_type=QuantType.QUInt8,
)

dyn_size_mb = onnx_dyn_int8_path.stat().st_size / (1024 * 1024)
dyn_model = onnx.load(str(onnx_dyn_int8_path))
dyn_ops = Counter(node.op_type for node in dyn_model.graph.node)

print(f"FP32 Size         : {onnx_fp32_size_mb:.2f} MB")
print(f"Dynamic INT8 Size : {dyn_size_mb:.2f} MB ({100*(1 - dyn_size_mb/onnx_fp32_size_mb):.1f}% smaller)")
print(f"Dynamic INT8 Ops  : {dict(dyn_ops)}")

### Surprise! Why Did Dynamic INT8 Barely Shrink `ResNet-18`?

Think of two different kinds of math blocks inside AI models:
- **`MatMul` / `Gemm`** (*Matrix Multiply*): Multiplying big tables of numbers. Text models & ChatGPT-style Transformers are mostly made of these.
- **`Conv`** (*Convolution*): A tiny magnifying glass sliding across an image to spot edges, fur, wheels, and eyes. Vision models (**CNNs**) like `ResNet-18` store **~95% of their size inside `Conv` layers**!

Look at the operator count above: `quantize_dynamic` **only quantized the 1 tiny final `Gemm` (`MatMul`) layer** and left all 20 big `Conv` layers in heavy FP32! That's because computing rounding scales on-the-fly for 2D image convolutions is slow on CPUs.

**Key Lesson (Huge Interview Insight):**
- Text/Transformer models -> Dynamic quantization works well right away.
- Image/CNN models (`ResNet`, `YOLO`, `MobileNet`) -> You need **Static INT8 Quantization with Calibration** to shrink `Conv` layers!

## 6. Shrinking Experiment B: Static INT8 with Real Calibration (`quantize_static` + QDQ)

### Simple Explanation (The Ruler Analogy)
Imagine a layer inside the AI outputs messy decimal numbers between `-2.0` and `+6.0`, and you want to store them on a tiny **8-bit integer ruler** that only has whole-number tick marks from `0` to `255`.

To map any decimal $r$ onto a ruler tick mark $q$, you only need **two numbers**:
1. **Scale ($S$):** How big is the gap between two neighboring tick marks on your ruler? $S = \frac{r_{\max} - r_{\min}}{255}$
2. **Zero-Point ($Z$):** Which integer tick mark on the `0..255` ruler represents exact `0.0`? (Super important because image padding and `ReLU` use lots of exact zeros!)

$$q = \text{clamp}\left(\text{round}\left(\frac{r}{S}\right) + Z,\ q_{\min},\ q_{\max}\right)$$

### Two More Terms Made Simple
- **Calibration (`CalibrationDataReader`):** How do we know a layer's real min/max (`-2.0` to `+6.0`)? We feed **100 real photos** through the FP32 model so ONNX Runtime can watch the actual numbers flowing through every layer and pick the best $S$ and $Z$ for each one.
- **QDQ Format (`QuantFormat.QDQ`):** Stands for **Quantize-Dequantize**. It inserts explicit `QuantizeLinear` (pack float -> INT8) and `DequantizeLinear` (unpack INT8 -> float) marker nodes around every layer. Why? Because it's the universal standard understood by **ONNX Runtime, NVIDIA TensorRT, and Qualcomm Snapdragon chips**!

In [ ]:
class ImageCalibrationReader(CalibrationDataReader):
    """Hands our 100 real photos one-by-one to ONNX Runtime so it can measure real number ranges."""

    def __init__(self, images: list[np.ndarray], input_name: str = "input"):
        self.images = images
        self.input_name = input_name
        self._iter = iter(self.images)

    def get_next(self):
        img = next(self._iter, None)
        if img is None:
            return None
        return {self.input_name: img}

    def rewind(self):
        self._iter = iter(self.images)


onnx_static_int8_path = ARTIFACT_DIR / "resnet18_static_int8_qdq.onnx"
calib_reader = ImageCalibrationReader(calib_images, input_name="input")

# Official ONNX Runtime recipe for x86 CPUs:
# QDQ format + Unsigned 8-bit activations (0..255) + Signed 8-bit weights (-128..127)
quantize_static(
    model_input=str(onnx_prep_path),
    model_output=str(onnx_static_int8_path),
    calibration_data_reader=calib_reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    calibrate_method=CalibrationMethod.MinMax,
)

static_size_mb = onnx_static_int8_path.stat().st_size / (1024 * 1024)
static_model = onnx.load(str(onnx_static_int8_path))
static_ops = Counter(node.op_type for node in static_model.graph.node)

print(f"FP32 Size              : {onnx_fp32_size_mb:.2f} MB")
print(f"Static INT8 (QDQ) Size : {static_size_mb:.2f} MB ({100*(1 - static_size_mb/onnx_fp32_size_mb):.1f}% smaller!)")
print(f"Static INT8 Ops        : {dict(static_ops)}")

## 7. Head-to-Head Benchmark: Size, Speed, and Accuracy

### Simple Explanation of Every Metric in Our Report
1. **Warmup Runs (20 un-timed runs):** Just like a sprinter stretches before the race, the first few AI runs are artificially slow while the computer sets up memory and caches. We always discard warmup runs before starting the stopwatch.
2. **Mean Latency (ms):** Average time in milliseconds to process **1 photo** (lower = faster).
3. **p95 Latency (ms):** The **95th percentile** speed — 95 out of 100 photos finish faster than this. Think of it as your "almost worst-case" speed. For smooth 30 FPS video (33.3 ms per frame), keeping `p95` low matters even more than average speed.
4. **FPS (Frames Per Second):** How many photos the model can inspect every second (`1000 / mean_ms`).
5. **Top-1 Agreement (%):** Out of 100 test photos, how often did the shrunk model pick the **exact same #1 answer** as the original full-size PyTorch model?
6. **Cosine Similarity (0 to 1):** Measures whether the shrunk model's 1,000 raw scores point in the exact same direction as FP32 (`1.000` = identical; `> 0.99` = virtually indistinguishable!).

In [ ]:
def benchmark_latency(run_fn, sample_input, warmup_runs: int = 20, timed_runs: int = 80) -> dict:
    """Time how fast a model runs 1 image after warming up the CPU."""
    for _ in range(warmup_runs):
        run_fn(sample_input)

    latencies_ms = []
    for _ in range(timed_runs):
        t0 = time.perf_counter()
        run_fn(sample_input)
        t1 = time.perf_counter()
        latencies_ms.append((t1 - t0) * 1000.0)

    arr = np.array(latencies_ms)
    mean_ms = float(np.mean(arr))
    return {
        "mean_ms": round(mean_ms, 2),
        "p50_ms": round(float(np.percentile(arr, 50)), 2),
        "p95_ms": round(float(np.percentile(arr, 95)), 2),
        "fps": round(1000.0 / mean_ms, 1),
    }


def evaluate_fidelity(run_fn, reference_logits: list[np.ndarray], images: list[np.ndarray]) -> dict:
    """Check how closely the compressed model agrees with the original FP32 model."""
    top1_matches = 0
    top5_matches = 0
    cos_sims = []
    maes = []

    for img, ref in zip(images, reference_logits):
        pred = run_fn(img)
        ref_1d = ref.flatten()
        pred_1d = pred.flatten()

        ref_top1 = int(np.argmax(ref_1d))
        pred_top1 = int(np.argmax(pred_1d))
        pred_top5 = np.argsort(pred_1d)[-5:]

        if pred_top1 == ref_top1:
            top1_matches += 1
        if ref_top1 in pred_top5:
            top5_matches += 1

        cos = float(np.dot(ref_1d, pred_1d) / (np.linalg.norm(ref_1d) * np.linalg.norm(pred_1d) + 1e-12))
        cos_sims.append(cos)
        maes.append(float(np.mean(np.abs(ref_1d - pred_1d))))

    n = len(images)
    return {
        "top1_agreement_pct": round(100.0 * top1_matches / n, 1),
        "top5_agreement_pct": round(100.0 * top5_matches / n, 1),
        "cosine_similarity": round(float(np.mean(cos_sims)), 5),
        "mean_abs_error": round(float(np.mean(maes)), 4),
    }


# Load ONNX Runtime sessions for both quantized models
ort_dyn_sess = create_ort_session(onnx_dyn_int8_path, num_threads=NUM_THREADS)
ort_static_sess = create_ort_session(onnx_static_int8_path, num_threads=NUM_THREADS)

def run_pytorch(x_np: np.ndarray) -> np.ndarray:
    with torch.no_grad():
        return pt_model(torch.from_numpy(x_np)).numpy()

runners = [
    ("PyTorch ResNet-18", "FP32", pt_size_mb, run_pytorch),
    ("ONNX Runtime", "FP32", onnx_fp32_size_mb, lambda x: ort_fp32_sess.run(None, {"input": x})[0]),
    ("ONNX Runtime", "Dynamic INT8", dyn_size_mb, lambda x: ort_dyn_sess.run(None, {"input": x})[0]),
    ("ONNX Runtime", "Static INT8 (QDQ)", static_size_mb, lambda x: ort_static_sess.run(None, {"input": x})[0]),
]

# Get the original PyTorch FP32 answers on all 100 test photos
print(f"Computing FP32 reference answers on {len(eval_images)} photos...")
ref_logits = [run_pytorch(img) for img in eval_images]

results = []
baseline_ms = None
baseline_size = pt_size_mb

for name, fmt, size_mb, fn in runners:
    print(f"Benchmarking {name} ({fmt})...")
    lat = benchmark_latency(fn, eval_images[0], warmup_runs=20, timed_runs=80)
    fid = evaluate_fidelity(fn, ref_logits, eval_images)
    if baseline_ms is None:
        baseline_ms = lat["mean_ms"]
    row = {
        "model": name,
        "format": fmt,
        "size_mb": round(size_mb, 2),
        "size_drop_pct": round(100.0 * (1.0 - size_mb / baseline_size), 1),
        "mean_ms": lat["mean_ms"],
        "p50_ms": lat["p50_ms"],
        "p95_ms": lat["p95_ms"],
        "fps": lat["fps"],
        "speedup": round(baseline_ms / lat["mean_ms"], 2),
        **fid,
    }
    results.append(row)

# Save JSON report so you can add it to your portfolio repo
report = {
    "hardware": {
        "cpu_cores": os.cpu_count(),
        "threads_used": NUM_THREADS,
        "torch_version": torch.__version__,
        "ort_version": ort.__version__,
        "input_shape": [1, 3, 224, 224],
        "calib_samples": CALIB_COUNT,
        "eval_samples": EVAL_COUNT,
    },
    "results": results,
}
report_path = ARTIFACT_DIR / "step01_benchmark_results.json"
report_path.write_text(json.dumps(report, indent=2))

# Print a clean Markdown table ready to paste into README.md
print("\n### Copy-Paste Ready Markdown Benchmark Table\n")
header = "| Model Variant | Format | Size (MB) | Size Drop | Mean Latency (ms) | p95 Latency (ms) | Speedup vs PyTorch | Top-1 Agreement vs FP32 | Cosine Similarity |"
sep    = "| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |"
print(header)
print(sep)
for r in results:
    print(
        f"| {r['model']} | {r['format']} | {r['size_mb']:.2f} | {r['size_drop_pct']:.1f}% | "
        f"{r['mean_ms']:.2f} | {r['p95_ms']:.2f} | {r['speedup']:.2f}x | "
        f"{r['top1_agreement_pct']:.1f}% | {r['cosine_similarity']:.5f} |"
    )

print(f"\nSaved full JSON report to: {report_path}")

## 8. Peek at Real Predictions: FP32 vs. Static INT8 Side-by-Side

### Simple Explanation
- **Softmax:** The AI outputs 1,000 raw scores (**logits**), like `[-2.1, 0.4, 8.7, ...]`. **Softmax** is simply a math converter that turns those raw scores into friendly **percentages (0% to 100%)** that all add up to 100%.
- Below, you can see the actual top object name and confidence % guessed by the original heavy model vs. your 4x smaller INT8 model.

In [ ]:
def softmax(x: np.ndarray) -> np.ndarray:
    """Convert raw scores (logits) into 0%..100% probabilities that sum to 1."""
    e = np.exp(x - np.max(x))
    return e / np.sum(e)

for idx in range(5):
    img = eval_images[idx]
    fp32_probs = softmax(ort_fp32_sess.run(None, {"input": img})[0].flatten())
    int8_probs = softmax(ort_static_sess.run(None, {"input": img})[0].flatten())

    fp32_id = int(np.argmax(fp32_probs))
    int8_id = int(np.argmax(int8_probs))
    match_tag = "MATCH" if fp32_id == int8_id else "DIFF "

    print(
        f"Photo #{idx} [{match_tag}] | "
        f"FP32: {IMAGENET_CLASSES[fp32_id]:20s} ({fp32_probs[fp32_id]*100:5.1f}%) | "
        f"INT8: {IMAGENET_CLASSES[int8_id]:20s} ({int8_probs[int8_id]*100:5.1f}%)"
    )

## 9. Two Quick Hands-On Experiments (Before Moving to Step 2!)

Try these two quick tweaks right inside Colab so the concepts truly click:

1. **Experiment 1 (`MinMax` vs. `Entropy` Calibration):**
   - **Simple Explanation:** Imagine measuring the heights of 100 students to build a 0–255 ruler. If 99 students are 150–180 cm tall, and 1 giant visitor is 300 cm tall:
     - **`MinMax`** stretches the whole ruler up to 300 cm just for that 1 outlier—squeezing the 99 normal students together and losing detail!
     - **`Entropy`** smartly clips rare extreme outliers so the 99 normal values stay super sharp.
   - **Try it:** In Section 6, change `CalibrationMethod.MinMax` to `CalibrationMethod.Entropy` and re-run Sections 6 & 7. Does `Top-1 Agreement` or `Cosine Similarity` improve?
2. **Experiment 2 (See Your AI Blueprint Visually):**
   - In Colab's left sidebar (folder icon), download `step01_artifacts/resnet18_static_int8_qdq.onnx` and drag-and-drop it into **[Netron.app](https://netron.app/)**.
   - Zoom into any `Conv` block and see the `QuantizeLinear -> DequantizeLinear -> Conv` pattern with your own eyes!

### When You Are Ready for Step 2, Share:
1. Your **Markdown Benchmark Table** output from Section 7.
2. One simple sentence in your own words: *Why did `quantize_dynamic` barely shrink `ResNet-18`, while `quantize_static` cut it by ~75%?*